# Three-LUT Unification — Encoder, Gate, Predictor

Mirrors `02_caal_cortex_unification.ipynb`.  In ds-hllset-cortex the three
LUTs held characters, bigrams, and trigrams.  Here they hold the three
generations of a **V-JEPA quantized encoding stream**:

- `LUT_1g` — unigram tids (the codebook vocabulary, the encoder's atoms)
- `LUT_2g` — bigram tids (adjacency structure — feeds De Bruijn order)
- `LUT_3g` — trigram tids (triadic structure — disambiguation)

The narrative source is `docs/EWM/main.tex` §HLLSet Algebra: TF is stored,
rank is derived, and the LUT is the only component that learns.

In [ ]:
import sys
import pathlib

ROOT = pathlib.Path.cwd()
if not (ROOT / "ewm_jepa").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import torch
from ewm_jepa import Quantizer
import hllset_py
from hllset_cortex import HLLSetFilter
from hllset_cortex.domain import encoding_tokenizer, hllset_from_ids, tid

# A deterministic tid-stream source: the same quantizer the JEPA pipeline
# uses, fed with synthetic encodings.  The EWM boundary is encoding-agnostic —
# these tids are interchangeable with V-JEPA quantized patch encodings.
def make_tid_stream(n=200, dim=1024, codebook_size=2048, seed=0):
    q = Quantizer(dim=dim, codebook_size=codebook_size, device="cpu", seed=seed)
    # decorrelate the data seed from the codebook seed so streams differ
    z = torch.randn(n, dim, generator=torch.Generator().manual_seed(1000 + seed))
    return q.ids_to_text(q.encode(z))


## 1. The three-LUT architecture

In [ ]:
from dataclasses import dataclass, field

@dataclass
class ThreeLUT:
    lut_1g: object = field(default_factory=hllset_py.TokenLut)
    lut_2g: object = field(default_factory=hllset_py.TokenLut)
    lut_3g: object = field(default_factory=hllset_py.TokenLut)

    def seed_1g(self, ids):
        self.lut_1g.record_all([f"tid{i}" for i in ids])
        return len(ids)

    def ingest(self, text):
        tok1 = encoding_tokenizer()
        toks = tok1.tokenize(text.encode())
        for t in toks:
            s = t.decode()
            if "\x00" not in s:
                self.lut_1g.record_all([s])
            elif s.count("\x00") == 1:
                self.lut_2g.record_all([s])
            else:
                self.lut_3g.record_all([s])
        return len(toks)

tlu = ThreeLUT()
tlu.seed_1g(range(2048))
text = make_tid_stream(n=80)
n = tlu.ingest(text)
print("ingested tokens:", n)
print("LUT sizes      :", tlu.lut_1g.len(), tlu.lut_2g.len(), tlu.lut_3g.len())


## 2. Cross-LUT disambiguation

When two tids collide in the 1-gram lattice, the 2-gram and 3-gram LUTs
resolve which one was actually adjacent — the bootstrap views vote.

In [ ]:
stream = make_tid_stream(n=60, seed=1)
tlu.ingest(stream)

# A colliding unigram pair: whichever tid has higher 2-gram support wins.
tids = stream.split()
from collections import Counter
bigram_support = Counter()
for a, b in zip(tids, tids[1:]):
    bigram_support[f"{a}\x00{b}"] += 1

print("distinct tids :", len(set(tids)))
print("bigram types  :", len(bigram_support))
print("top bigrams   :", list(bigram_support.items())[:3])


## 3. De Bruijn order restoration (ungated bigram topology)

Exactly as in the original: the bigram HLLSet is **not** gated, because the
gate is built from unigrams and would destroy the bigram topology.  Order
restoration is the Type-1 forward model.

In [ ]:
db_tok = (
    hllset_py.Tokenizer()
    .pattern(list((__import__("string").ascii_lowercase + __import__("string").digits + "_-").encode()))
    .lowercase()
    .pad(b"<S>", b"</S>")
    .ngrams(2, 2)
)

def debruijn_restore(text, lut2):
    bigrams = db_tok.tokenize(text.encode())
    h_db = hllset_py.HLLSet.from_token_bytes(bigrams)
    lut2.record_all_bytes(bigrams)
    ordered = hllset_py.materialize_debruijn(h_db, lut2, "<S>", "</S>")
    payload = [t if isinstance(t, str) else t.decode() for t in ordered
               if t not in ("<S>", "</S>")]
    return payload

short = " ".join(make_tid_stream(n=12, seed=2).split()[:12])
restored = debruijn_restore(short, tlu.lut_2g)
print("original:", short.split())
print("restored:", restored)
print("exact   :", restored == short.split())


## 4. Content-addressed retrieval — BSS over documents

Documents are HLLSets; similarity is the directed BSS pair.  Two quantized
"documents" from the same seed share structure; different seeds do not.

In [ ]:
d1 = make_tid_stream(n=120, seed=0)
d2 = make_tid_stream(n=120, seed=0)   # same underlying structure
d3 = make_tid_stream(n=120, seed=99)  # different structure

h1 = hllset_py.HLLSet.from_tokens(d1.split())
h2 = hllset_py.HLLSet.from_tokens(d2.split())
h3 = hllset_py.HLLSet.from_tokens(d3.split())

def tau(a, b): return a.intersection(b).popcount() / max(b.popcount(), 1)
def rho(a, b): return a.difference(b).popcount() / max(b.popcount(), 1)

for name, hx in [("same", h2), ("different", h3)]:
    print(f"{name:>9}: tau={tau(h1, hx):.3f} rho={rho(h1, hx):.3f}")


## 5. Summary

The three-LUT architecture from ds-hllset-cortex carries over unchanged to
V-JEPA encodings.  The only model-specific piece is the **token definition**
(`tid{n}` from the quantizer) — exactly as `main.tex` §5.7 predicted:
*"What changes is the token definition — what string you feed to the hash
function."*